In [1]:
import sys
import numpy as np
import pubchempy as pcp
from rdkit import Chem, DataStructs
from rdkit.Chem import rdFingerprintGenerator
from tensorflow.keras.models import load_model

def get_smiles_from_iupac(iupac_name):
    """
    Retrieve the canonical SMILES for a compound using its IUPAC name.
    This function queries PubChem.
    """
    compounds = pcp.get_compounds(iupac_name, 'name')
    if compounds:
        return compounds[0].canonical_smiles
    else:
        raise ValueError("Compound not found in PubChem.")

def smiles_to_fingerprint(smiles, radius=2, fpSize=2048):
    """
    Convert a SMILES string to an RDKit Morgan fingerprint using the
    rdFingerprintGenerator module.
    
    This uses:
        generator = rdFingerprintGenerator.GetMorganGenerator(radius, fpSize)
        fingerprint = generator.GetFingerprint(mol)
    
    The fingerprint is then converted into a NumPy array suitable for model input.
    """
    # Create the molecule from the SMILES string
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        raise ValueError("Could not generate a molecule from the SMILES string.")
    
    # Initialize the Morgan generator with the specified parameters
    generator = rdFingerprintGenerator.GetMorganGenerator(radius=radius, fpSize=fpSize)
    
    # Generate the fingerprint
    fp = generator.GetFingerprint(mol)
    
    # Convert the fingerprint to a NumPy array
    fp_array = np.zeros((fpSize,), dtype=int)
    DataStructs.ConvertToNumpyArray(fp, fp_array)
    
    # Reshape for model input (1 sample, nFeatures)
    return fp_array.reshape(1, -1)

def main():
    # Step 1: Get IUPAC name input from the user
    iupac_name = input("Enter the IUPAC name of the compound: ").strip()
    
    try:
        # Step 2: Convert IUPAC name to SMILES using PubChem
        smiles = get_smiles_from_iupac(iupac_name)
        print(f"Converted SMILES: {smiles}")
    except Exception as e:
        print("Error converting IUPAC name to SMILES:", e)
        sys.exit(1)
    
    try:
        # Step 3: Generate the Morgan fingerprint using rdFingerprintGenerator
        fingerprint = smiles_to_fingerprint(smiles)
        print("Generated fingerprint (as numpy array):")
        print(fingerprint)
    except Exception as e:
        print("Error generating fingerprint:", e)
        sys.exit(1)
    
    # (Optional) If you have a pre-trained pKa prediction model saved in a .h5 file:
    try:
        model = load_model("best_model_CatBoost.pkl")
    except Exception as e:
        print("Error loading the prediction model:", e)
        sys.exit(1)
    
    try:
        # Step 4: Use the model to predict the pKa from the fingerprint
        predicted_pka = model.predict(fingerprint)
        # Adjust output based on the shape of the prediction
        if predicted_pka.ndim == 2 and predicted_pka.shape[1] == 1:
            predicted_value = predicted_pka[0, 0]
        else:
            predicted_value = predicted_pka[0]
        print(f"Predicted pKa value: {predicted_value}")
    except Exception as e:
        print("Error during prediction:", e)
        sys.exit(1)

if __name__ == "__main__":
    main()


Enter the IUPAC name of the compound:  ethanol


Converted SMILES: CCO
Generated fingerprint (as numpy array):
[[0 0 0 ... 0 0 0]]
Error loading the prediction model: File format not supported: filepath=best_model_CatBoost.pkl. Keras 3 only supports V3 `.keras` files and legacy H5 format files (`.h5` extension). Note that the legacy SavedModel format is not supported by `load_model()` in Keras 3. In order to reload a TensorFlow SavedModel as an inference-only layer in Keras 3, use `keras.layers.TFSMLayer(best_model_CatBoost.pkl, call_endpoint='serving_default')` (note that your `call_endpoint` might have a different name).


AttributeError: 'tuple' object has no attribute 'tb_frame'

In [41]:
from rdkit import Chem
from rdkit.Chem import rdFingerprintGenerator

# Create your molecule from a SMILES string
mol = Chem.MolFromSmiles("OC1CCCCC1")  # cyclohexanol

# Initialize the Morgan generator with your desired parameters
generator = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)

# Generate the fingerprint
fingerprint = generator.GetFingerprint(mol)
